# AIFS Forecast from Brightband Initial Conditions

Runs ECMWF's [aifs-single-1.0](https://huggingface.co/ecmwf/aifs-single-1.0) model using
Brightband's [ECMWF IFS Initial Conditions](https://app.earthmover.io/marketplace/697162921880507a6587c31b)
from the Earthmover data marketplace, and writes the forecast back to Arraylake.

In [ ]:
import datetime
import threading
import queue

import numpy as np
import pandas as pd
import torch

from arraylake import Client
from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from main import (
    DEFAULT_IC_REPO,
    open_initial_conditions,
    fetch_initial_conditions,
    get_gpu_regridder,
    state_to_xarray,
    datetime_to_str,
)

In [ ]:
client = Client()
client.login()

## Open the initial conditions from the marketplace subscription

In [ ]:
ds, ds_static = open_initial_conditions(DEFAULT_IC_REPO)
ds

In [ ]:
# forecast from the most recent available analysis
date = pd.Timestamp(ds.init_time.values[-1]).to_pydatetime().replace(tzinfo=datetime.UTC)
print("Initial date is", date)

## Fetch and regrid the input fields (0.25° ➞ N320, on the GPU)

In [ ]:
input_regridder = get_gpu_regridder({"grid": (0.25, 0.25)}, {"grid": "N320"})
%time fields = fetch_initial_conditions(date, ds, ds_static, input_regridder)

## Load the model

In [ ]:
checkpoint = {"huggingface": "ecmwf/aifs-single-1.0"}
runner = SimpleRunner(checkpoint, device="cuda")

In [ ]:
target_repo = client.get_or_create_repo("earthmover-public/aifs-outputs")
target_session = target_repo.writable_session("main")

## Run the forecast

Outputs are regridded back to 0.25° on the GPU and written to Arraylake from a background thread.

In [ ]:
output_regridder = get_gpu_regridder({"grid": "N320"}, {"grid": (0.25, 0.25)})

date_no_tz = date.replace(tzinfo=None)
input_state = dict(date=date_no_tz, fields=fields)

# we put data that we want to write into a queue
q = queue.Queue()
lock = threading.Lock()

def worker():
    while True:
        (ds_out, store, group_name, kwargs) = q.get()
        # lock is probably unncessary
        with lock:
            ds_out.to_zarr(
                store, group=group_name, zarr_format=3, consolidated=False, **kwargs
            )
        q.task_done()

# a separate thread for I/O to avoid blocking the main loop
threading.Thread(target=worker, daemon=True).start()

print("starting forecast loop")
kwargs = {"mode": "w"}

# clear GPU memory
torch.cuda.empty_cache()

# main forecast loop
for n, state in enumerate(runner.run(input_state=input_state, lead_time=96)):
    print_state(state)
    ds_out = state_to_xarray(state, regridder=output_regridder).chunk()
    group = datetime_to_str(date)
    if n > 0:
        kwargs = {"mode": "a", "append_dim": "valid_time"}
    q.put((ds_out, target_session.store, group, kwargs))

q.join()  # wait for all I/O tasks to finish

# clear GPU memory
torch.cuda.empty_cache()

In [ ]:
target_session.commit(f"Wrote a 96 hour forecast for {date}")